# TP Seance 2 - Amelioration des Performances et Generalisation

**Objectifs :**
1. Diagnostiquer l'overfitting / underfitting a partir des courbes d'apprentissage
2. Reguler avec le Dropout (classique et Spatial)
3. Mettre en place l'Early Stopping (`restore_best_weights`)
4. Comparer plusieurs strategies de Learning Rate Scheduling
5. Analyse critique : matrice de confusion, metriques par classe, erreurs confiantes
6. Estimer l'incertitude avec MC Dropout + reject option
7. Valider statistiquement un gain (t-test, bootstrap)

**Dataset :** Fashion-MNIST (10 classes).

> **Fil conducteur :** un seul builder `build_cnn(regularized=...)`, un seul helper
> `train(...)`. On ne change qu'**un facteur a la fois** pour des comparaisons honnetes
> (ex. Partie 2 : overfit vs +Dropout, **meme architecture**, seul le Dropout change).

## TensorBoard

Pour visualiser les courbes pendant/apres l'entrainement, depuis un terminal :

```
pip install tensorboard
tensorboard --logdir logs/
```

Ou directement dans le notebook (derniere cellule du TP).

---
## Partie 0 - Setup, donnees et briques reutilisables

In [ ]:
import datetime
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                        ReduceLROnPlateau, LearningRateScheduler, TensorBoard)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from scipy import stats

# Reproductibilite
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

In [ ]:
# Callback TensorBoard (defini UNE seule fois). histogram_freq=0 -> leger et rapide.
def make_tensorboard_cb(name):
    log_dir = f"logs/{name}_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    return TensorBoard(log_dir=log_dir, histogram_freq=0)

In [ ]:
# Chargement de Fashion-MNIST
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
CLASS_NAMES = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
               'Sandale', 'Chemise', 'Basket', 'Sac', 'Botte']
print(f"Train {X_train_full.shape} | Test {X_test.shape}")

In [ ]:
# Split AVANT normalisation, puis normalisation fit sur le TRAIN uniquement (pas de data leakage)
X_val,   y_val   = X_train_full[:5000], y_train_full[:5000]
X_train, y_train = X_train_full[5000:], y_train_full[5000:]

mu, sigma = X_train.mean(), X_train.std()          # statistiques du TRAIN seulement
X_train = ((X_train - mu) / sigma)[..., np.newaxis]
X_val   = ((X_val   - mu) / sigma)[..., np.newaxis]   # memes mu/sigma
X_test  = ((X_test  - mu) / sigma)[..., np.newaxis]   # memes mu/sigma

print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")
print("Normalisation fit sur train uniquement -> pas de data leakage.")

In [ ]:
# Apercu
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax in axes.flat:
    idx = np.random.randint(len(X_train))
    ax.imshow(X_train[idx].squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[idx]], fontsize=10); ax.axis('off')
plt.suptitle('Echantillons Fashion-MNIST'); plt.tight_layout(); plt.show()

### Les briques reutilisables

- **`build_cnn(regularized=...)`** : meme architecture ; `regularized=True` ajoute
  `SpatialDropout2D` (conv) et `Dropout` (dense). Remplace les deux fonctions `build_*` du TP d'origine.
- **`train(model, name, ...)`** : compile (une fois) + TensorBoard + fit.
- **`plot_diagnostic(history)`** : courbes train/val + verdict automatique du regime.

In [ ]:
EPOCHS     = 20            # entrainements "normaux"
BATCH_SIZE = 64            # identique partout -> comparaisons equitables
MAX_EPOCHS = 100           # cap pour l'Early Stopping (c'est le callback qui arrete)
INPUT_SHAPE = (28, 28, 1)


def build_cnn(regularized=False, input_shape=INPUT_SHAPE, num_classes=10):
    """Meme CNN sur-dimensionne. regularized=True -> SpatialDropout2D + Dropout."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filters in (128, 256):
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.MaxPooling2D()(x)
        if regularized:
            x = layers.SpatialDropout2D(0.25)(x)        # dropout spatial pour les conv
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    if regularized:
        x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    if regularized:
        x = layers.Dropout(0.5)(x)                       # jamais de Dropout APRES le softmax
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs,
                       name='CNN_regularized' if regularized else 'CNN_overfit')


def train(model, name, epochs=EPOCHS, extra_callbacks=None, verbose=1):
    """Compile (une seule fois) + TensorBoard + fit. batch_size constant."""
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cbs = [make_tensorboard_cb(name)] + (extra_callbacks or [])
    return model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=epochs, batch_size=BATCH_SIZE, callbacks=cbs, verbose=verbose)


def plot_diagnostic(history, title=""):
    """Courbes train/val + diagnostic automatique du regime."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(history.history['loss'], label='train', linewidth=2)
    a1.plot(history.history['val_loss'], '--', label='val', linewidth=2)
    a1.set_title('Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
    a2.plot(history.history['accuracy'], label='train', linewidth=2)
    a2.plot(history.history['val_accuracy'], '--', label='val', linewidth=2)
    a2.set_title('Accuracy'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
    if title:
        fig.suptitle(title, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

    gap = history.history['val_loss'][-1] - history.history['loss'][-1]
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    if train_acc < 0.85:
        regime = "Underfitting probable (le modele n'apprend pas assez)"
    elif gap > 0.15:
        regime = "Overfitting (variance elevee : grand ecart train/val)"
    else:
        regime = "Bon ajustement"
    print(f"acc train={train_acc:.2%} | acc val={val_acc:.2%} | gap(val-train loss)={gap:.4f}")
    print("Regime :", regime)
    return gap

---
## Partie 1 - Provoquer volontairement l'overfitting

Un modele **sur-dimensionne sans regularisation** : on s'attend a un grand ecart train/val.

In [ ]:
model_overfit = build_cnn(regularized=False)
model_overfit.summary()

In [ ]:
history_overfit = train(model_overfit, "01_sans_regularisation")

In [ ]:
gap_overfit = plot_diagnostic(history_overfit, "Sans regularisation")

### Question 1.1
- Interpretez les deux courbes (Loss et Accuracy).
- A quelle epoch l'overfitting devient-il visible ?
- Quel est le gap train/val final ? Dans quel regime biais-variance est ce modele ?

**Votre reponse :**

...

---
## Partie 2 - Regularisation par Dropout

Meme architecture, on **ajoute seulement le Dropout** (`build_cnn(regularized=True)`).
Comme seul ce facteur change, l'ecart de comportement est imputable au Dropout.

In [ ]:
model_reg = build_cnn(regularized=True)
model_reg.summary()

In [ ]:
history_reg = train(model_reg, "02_avec_dropout")

In [ ]:
gap_reg = plot_diagnostic(history_reg, "Avec Dropout")

In [ ]:
# Comparaison directe overfit vs Dropout
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(history_overfit.history['val_loss'], color='#C62828', label='sans regularisation', linewidth=2)
a1.plot(history_reg.history['val_loss'],     color='#2E7D32', label='avec Dropout',        linewidth=2)
a1.set_title('Validation Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.plot(history_overfit.history['val_accuracy'], color='#C62828', label='sans regularisation', linewidth=2)
a2.plot(history_reg.history['val_accuracy'],     color='#2E7D32', label='avec Dropout',        linewidth=2)
a2.set_title('Validation Accuracy'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.suptitle('Effet du Dropout', fontweight='bold', y=1.02); plt.tight_layout(); plt.show()

print(f"Gap sans regularisation : {gap_overfit:.4f}")
print(f"Gap avec Dropout        : {gap_reg:.4f}")
print(f"Reduction du gap        : {(gap_overfit - gap_reg) / gap_overfit:.0%}")

### Question 2.1
- Le Dropout a-t-il reduit le gap train/val ?
- Pourquoi `SpatialDropout2D` plutot que `Dropout` apres les couches convolutionnelles ?
- Pourquoi ne met-on pas de Dropout juste avant le softmax ?

**Votre reponse :**

...

---
## Partie 3 - Early Stopping

- `epochs` volontairement **grand** : c'est le callback qui arrete.
- **Toujours** `restore_best_weights=True`.

In [ ]:
model_es = build_cnn(regularized=True)
early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-4,
                           restore_best_weights=True, verbose=1)
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss',
                             save_best_only=True, verbose=0)

history_es = train(model_es, "03_early_stopping", epochs=MAX_EPOCHS,
                   extra_callbacks=[early_stop, checkpoint])

In [ ]:
gap_es = plot_diagnostic(history_es, "Dropout + Early Stopping")

stopped = len(history_es.history['loss'])
best    = int(np.argmin(history_es.history['val_loss'])) + 1
print(f"Arret a l'epoch {stopped} | meilleur modele a l'epoch {best} "
      f"| meilleure val_loss = {min(history_es.history['val_loss']):.4f}")
print("Meilleur modele restaure automatiquement (restore_best_weights=True).")

### Question 3.1
- A quelle epoch l'entrainement s'est-il arrete ? Le modele s'ameliorait-il encore ?
- Que se passerait-il avec `restore_best_weights=False` ?
- Pourquoi augmenter la patience quand on utilise de la Data Augmentation ?

**Votre reponse :**

...

---
## Partie 4 - Learning Rate Scheduling

On compare 4 schedules + `ReduceLROnPlateau`, toutes choses egales par ailleurs.

In [ ]:
TOTAL_EPOCHS = 30
LR_MAX, LR_MIN = 1e-3, 1e-6

def cosine_schedule(epoch):
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + np.cos(np.pi * epoch / TOTAL_EPOCHS))

def step_decay_schedule(epoch):
    return LR_MAX * (0.5 ** (epoch // 10))

def warmup_cosine_schedule(epoch, warmup=5):
    if epoch < warmup:
        return LR_MAX * (epoch + 1) / warmup
    progress = (epoch - warmup) / (TOTAL_EPOCHS - warmup)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + np.cos(np.pi * progress))

# Visualisation des schedules
e = np.arange(TOTAL_EPOCHS)
plt.figure(figsize=(9, 4))
plt.plot(e, [LR_MAX] * TOTAL_EPOCHS, label='constant')
plt.plot(e, [step_decay_schedule(i) for i in e], label='step decay')
plt.plot(e, [cosine_schedule(i) for i in e], label='cosine')
plt.plot(e, [warmup_cosine_schedule(i) for i in e], label='warmup + cosine')
plt.xlabel('epoch'); plt.ylabel('learning rate'); plt.legend(); plt.grid(alpha=.3)
plt.title('Learning rate schedules'); plt.tight_layout(); plt.show()

In [ ]:
def train_with_scheduler(name, sched_callbacks):
    tf.random.set_seed(42)                       # meme initialisation pour comparer
    model = build_cnn(regularized=True)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
    slug = name.lower().replace(' ', '_').replace('+', 'plus')
    h = train(model, f"04_{slug}", epochs=TOTAL_EPOCHS,
              extra_callbacks=sched_callbacks + [es], verbose=0)
    print(f"{name:20s} | val_loss={min(h.history['val_loss']):.4f} "
          f"| val_acc={max(h.history['val_accuracy']):.2%} | epochs={len(h.history['loss'])}")
    return h, model

print(f"{'Scheduler':20s} | {'val_loss':9s} | {'val_acc':8s} | epochs")
print('-' * 60)
lr_runs = {}
lr_runs['Constant']        = train_with_scheduler('Constant', [])
lr_runs['Step Decay']      = train_with_scheduler('Step Decay', [LearningRateScheduler(step_decay_schedule)])
lr_runs['Cosine']          = train_with_scheduler('Cosine', [LearningRateScheduler(cosine_schedule)])
lr_runs['Warmup+Cosine']   = train_with_scheduler('Warmup Cosine', [LearningRateScheduler(warmup_cosine_schedule)])
lr_runs['ReduceOnPlateau'] = train_with_scheduler('ReduceOnPlateau',
    [ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=0)])

In [ ]:
# Courbes comparees
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5))
for name, (h, _) in lr_runs.items():
    a1.plot(h.history['val_loss'], label=name, linewidth=2)
    a2.plot(h.history['val_accuracy'], label=name, linewidth=2)
a1.set_title('Validation Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.set_title('Validation Accuracy'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.suptitle('Comparaison des schedulers', fontweight='bold', y=1.02); plt.tight_layout(); plt.show()

### Question 4.1
- Quel scheduler donne les meilleurs resultats ? Pourquoi ?
- Pourquoi le warmup est-il critique pour les Transformers et moins pour les CNN ?
- Dans quel cas recommander `ReduceLROnPlateau` en entreprise ?

**Votre reponse :**

...

---
## Partie 5 - Analyse critique des performances

On selectionne **automatiquement** le meilleur scheduler (val accuracy) et on l'evalue sur le test (une seule fois).

In [ ]:
best_name = max(lr_runs, key=lambda n: max(lr_runs[n][0].history['val_accuracy']))
best_model = lr_runs[best_name][1]
print("Meilleur modele :", best_name)

y_proba = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_proba, axis=1)
y_true = y_test.flatten()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Matrice de confusion - test'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

cm_err = cm.copy(); np.fill_diagonal(cm_err, 0)
print("Top 5 des confusions :")
for r in range(5):
    i, j = np.unravel_index(cm_err.argmax(), cm_err.shape)
    print(f"  {CLASS_NAMES[i]:>10} -> {CLASS_NAMES[j]:<10} : {cm_err[i, j]} erreurs")
    cm_err[i, j] = 0

### Analyse des erreurs les plus confiantes
*"Une erreur a 99 % de confiance est plus dangereuse qu'une erreur a 55 %."*

In [ ]:
conf = y_proba.max(axis=1)
wrong = np.where(y_pred != y_true)[0]
worst = wrong[np.argsort(-conf[wrong])]            # erreurs triees par confiance decroissante
print(f"Erreurs : {len(wrong)} / {len(y_true)} ({len(wrong)/len(y_true):.2%})")

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, idx in zip(axes.flat, worst[:12]):
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    ax.set_title(f"pred {CLASS_NAMES[y_pred[idx]]}\nvrai {CLASS_NAMES[y_true[idx]]}\n{conf[idx]:.0%}",
                 fontsize=8, color='#C62828')
    ax.axis('off')
plt.suptitle('Erreurs les plus confiantes', fontweight='bold'); plt.tight_layout(); plt.show()

### Question 5.1
- Quelles 2 classes sont les plus confondues ? Est-ce logique visuellement ?
- Les erreurs confiantes : bruit, ambiguite, ou erreurs de label ?
- En production, quelle action face a ces erreurs confiantes ?

**Votre reponse :**

...

---
## Partie 6 - MC Dropout : estimer l'incertitude

On garde le Dropout **actif a l'inference** (`training=True`) pour T predictions, dont on tire
la moyenne (prediction) et la variance (incertitude **epistemique**).

In [ ]:
def mc_dropout_predict(model, X, T=50):
    preds = np.stack([model(X, training=True).numpy() for _ in range(T)])
    return preds.mean(axis=0), preds.var(axis=0)

sample_X, sample_y = X_test[:600], y_test[:600]
mean_pred, var_pred = mc_dropout_predict(best_model, sample_X, T=50)

mc_classes      = np.argmax(mean_pred, axis=1)
mc_confidences  = mean_pred.max(axis=1)
mc_uncertainty  = var_pred.max(axis=1)
print("MC Dropout termine (50 forward passes).")

In [ ]:
print(f"{'#':>3} {'classe':>10} {'confiance':>10} {'incertitude':>12} {'ok':>4}")
print('-' * 45)
for i in range(15):
    ok = 'OK' if mc_classes[i] == sample_y[i] else 'X'
    print(f"{i:3d} {CLASS_NAMES[mc_classes[i]]:>10} {mc_confidences[i]:>10.2%} "
          f"{mc_uncertainty[i]:>12.6f} {ok:>4}")

In [ ]:
correct = mc_classes == sample_y.flatten()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.hist(mc_uncertainty[correct],  bins=30, alpha=.6, color='#2E7D32', label='corrects')
a1.hist(mc_uncertainty[~correct], bins=30, alpha=.6, color='#C62828', label='incorrects')
a1.set_title('Distribution de l incertitude'); a1.set_xlabel('incertitude (var max)'); a1.legend(); a1.grid(alpha=.3)
a2.scatter(mc_confidences[correct],  mc_uncertainty[correct],  s=20, alpha=.5, c='#2E7D32', label='corrects')
a2.scatter(mc_confidences[~correct], mc_uncertainty[~correct], s=40, alpha=.8, c='#C62828', marker='x', label='incorrects')
a2.set_title('Confiance vs incertitude'); a2.set_xlabel('confiance'); a2.set_ylabel('incertitude'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Reject option
En production, au-dela d'un **seuil d'incertitude**, la prediction est rejetee (renvoyee a un humain).

In [ ]:
thresholds = np.linspace(0, mc_uncertainty.max(), 50)
acc, rej = [], []
for t in thresholds:
    keep = mc_uncertainty <= t
    acc.append((mc_classes[keep] == sample_y.flatten()[keep]).mean() if keep.sum() else 1.0)
    rej.append(1 - keep.mean())

fig, ax1 = plt.subplots(figsize=(10, 5)); ax2 = ax1.twinx()
ax1.plot(thresholds, acc, 'b-', linewidth=2, label='accuracy (acceptees)')
ax2.plot(thresholds, rej, 'r--', linewidth=2, label='taux de rejet')
ax1.set_xlabel('seuil incertitude'); ax1.set_ylabel('accuracy', color='b'); ax2.set_ylabel('taux de rejet', color='r')
ax1.legend(loc='center left'); ax2.legend(loc='center right')
plt.title('Compromis accuracy vs rejet'); ax1.grid(alpha=.3); plt.tight_layout(); plt.show()

# Plus grand seuil atteignant 90 % d'accuracy (= on accepte le plus possible)
target = 0.90
ok_idx = [i for i, a in enumerate(acc) if a >= target]
if ok_idx:
    i = ok_idx[-1]
    print(f"Pour >= {target:.0%} d'accuracy : seuil={thresholds[i]:.6f}, rejet={rej[i]:.1%}")
else:
    print(f"Cible {target:.0%} non atteignable.")

### Question 6.1
- Les erreurs ont-elles en moyenne une incertitude plus elevee ?
- Interet de la reject option en contexte medical / financier ?
- Avantages de MC Dropout vs d'autres methodes d'incertitude ?
- Recommanderiez-vous de deployer ce modele ?

**Votre reponse :**

...

---
## Partie 7 - Validation statistique

On compare **le meme builder** avec et sans Dropout, sur plusieurs seeds, puis on teste la
significativite (t-test apparie + bootstrap). *"Un gain de 0.3 % non significatif ne justifie pas un redeploiement."*

In [ ]:
def multi_runs(regularized, n_runs=5, epochs=15):
    accs = []
    for run in range(n_runs):
        tf.random.set_seed(run)
        m = build_cnn(regularized=regularized)
        m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=epochs, batch_size=BATCH_SIZE,
              callbacks=[EarlyStopping(monitor='val_loss', patience=8,
                                       restore_best_weights=True, verbose=0)], verbose=0)
        a = m.evaluate(X_test, y_test, verbose=0)[1]
        accs.append(a); print(f"  run {run+1}/{n_runs} : {a:.4f}")
    return np.array(accs)

print("Modele A (sans Dropout) :"); acc_A = multi_runs(False)
print("Modele B (avec Dropout) :"); acc_B = multi_runs(True)

In [ ]:
print(f"A (sans Dropout) : {acc_A.mean():.4f} +/- {acc_A.std():.4f}")
print(f"B (avec Dropout) : {acc_B.mean():.4f} +/- {acc_B.std():.4f}")

# t-test apparie
t_stat, p = stats.ttest_rel(acc_B, acc_A)
print(f"\nt-test apparie : t={t_stat:.3f}, p={p:.4f}")
print("=> difference significative (p<0.05)" if p < 0.05 else "=> difference NON significative")

diff = acc_B - acc_A
ci = stats.t.interval(0.95, len(diff) - 1, loc=diff.mean(), scale=stats.sem(diff))
print(f"Difference moyenne : {diff.mean():.4f}  IC95% = [{ci[0]:.4f}, {ci[1]:.4f}]")

In [ ]:
# Bootstrap sur la difference appariee
np.random.seed(42)
boot = np.array([np.random.choice(diff, len(diff), replace=True).mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Bootstrap (10000) : difference={diff.mean():.4f}  IC95% = [{lo:.4f}, {hi:.4f}]")
print("=> IC ne contient pas 0 : significatif" if (lo > 0 or hi < 0) else "=> IC contient 0 : non significatif")

### Question 7.1
- La difference est-elle significative ?
- Pourquoi plusieurs runs plutot qu'un seul resultat ?
- Comment presenter ces resultats dans un rapport professionnel ?

**Votre reponse :**

...

---
## Partie 8 - Synthese

Tableau recapitulatif **genere automatiquement** a partir des experiences.

In [ ]:
def best_row(name, history):
    h = history.history
    k = int(np.argmin(h['val_loss']))                     # epoch de meilleure val_loss
    gap = h['val_loss'][k] - h['loss'][k]
    return (name, h['val_accuracy'][k], h['val_loss'][k], gap, len(h['loss']))

rows = [best_row('Sans regularisation', history_overfit),
        best_row('+ Dropout', history_reg),
        best_row('+ Dropout + EarlyStopping', history_es)]
rows += [best_row(f'+ LR {n}', h) for n, (h, _) in lr_runs.items()]

print(f"{'Experience':30s}{'val_acc':>10}{'val_loss':>10}{'gap':>10}{'epochs':>8}")
print('-' * 68)
for name, va, vl, gap, ep in rows:
    print(f"{name:30s}{va:>10.4f}{vl:>10.4f}{gap:>10.4f}{ep:>8d}")

### Checklist de l'ingenieur Deep Learning

**Avant :** split sans leakage | normalisation fit sur train | baseline simple.
**Pendant :** Early Stopping (`restore_best_weights=True`) | LR scheduler | Dropout / regularisation.
**Apres :** courbes analysees | matrice de confusion | erreurs confiantes | test statistique | test **une seule fois**.

**Regle d'or :** le jeu de test ne sert qu'une seule fois, a la toute fin.

### Visualiser avec TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/